# Practice 3 - TV4: Pretrained Model and Tokenizer

## Responsibility
Load a pretrained model and tokenizer suitable for binary text classification.

## Sections
- Select pretrained model
- Load tokenizer
- Load sequence classification model
- Verify model/tokenizer compatibility
- Prepare objects for training


In [6]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# ===== CẤU HÌNH CHUNG =====

DATASET_NAME = "stanfordnlp/imdb"

# Model tạm thời.
# Khi nhóm chốt model khác, chỉ cần thay đổi MODEL_NAME.
MODEL_NAME = "distilbert-base-uncased"

NUM_LABELS = 2

ID2LABEL = {
    0: "Negative",
    1: "Positive",
}

LABEL2ID = {
    "Negative": 0,
    "Positive": 1,
}

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("===== CẤU HÌNH =====")
print("Dataset    :", DATASET_NAME)
print("Model      :", MODEL_NAME)
print("Num labels :", NUM_LABELS)
print("Device     :", device)

===== CẤU HÌNH =====
Dataset    : stanfordnlp/imdb
Model      : distilbert-base-uncased
Num labels : 2
Device     : cpu


In [7]:
# Load pretrained tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("===== TOKENIZER =====")
print("Model name      :", MODEL_NAME)
print("Tokenizer class :", tokenizer.__class__.__name__)
print("Vocabulary size :", tokenizer.vocab_size)
print("Max length      :", tokenizer.model_max_length)

===== TOKENIZER =====
Model name      : distilbert-base-uncased
Tokenizer class : BertTokenizer
Vocabulary size : 30522
Max length      : 512


In [8]:
# Load pretrained model for binary classification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

model = model.to(device)

print("===== PRETRAINED MODEL =====")
print("Model name  :", MODEL_NAME)
print("Model class :", model.__class__.__name__)
print("Num labels  :", model.config.num_labels)
print("Device      :", next(model.parameters()).device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5885.50it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


===== PRETRAINED MODEL =====
Model name  : distilbert-base-uncased
Model class : DistilBertForSequenceClassification
Num labels  : 2
Device      : cpu


In [11]:
# Verify tokenizer and model forward pass

sample_text = "This movie is very interesting and enjoyable."

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
)

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

print("===== TOKENIZER TEST =====")
print("Sample text    :", sample_text)
print("Tokens         :", tokens)
print("Input IDs      :", inputs["input_ids"][0].tolist())
print("Attention Mask :", inputs["attention_mask"][0].tolist())

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

print("\n===== MODEL TEST =====")
print("Input shape  :", inputs["input_ids"].shape)
print("Logits       :", outputs.logits)
print("Logits shape :", outputs.logits.shape)

===== TOKENIZER TEST =====
Sample text    : This movie is very interesting and enjoyable.
Tokens         : ['[CLS]', 'this', 'movie', 'is', 'very', 'interesting', 'and', 'enjoyable', '.', '[SEP]']
Input IDs      : [101, 2023, 3185, 2003, 2200, 5875, 1998, 22249, 1012, 102]
Attention Mask : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

===== MODEL TEST =====
Input shape  : torch.Size([1, 10])
Logits       : tensor([[0.0404, 0.0239]])
Logits shape : torch.Size([1, 2])


In [10]:
# Inspect tokenizer output

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0].cpu()
)

print("===== TOKENIZER TEST =====")
print("Input IDs      :", inputs["input_ids"][0].cpu().tolist())
print("Attention Mask :", inputs["attention_mask"][0].cpu().tolist())
print("Tokens         :", tokens)

===== TOKENIZER TEST =====
Input IDs      : [101, 2023, 3185, 2003, 2200, 5875, 1998, 22249, 1012, 102]
Attention Mask : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Tokens         : ['[CLS]', 'this', 'movie', 'is', 'very', 'interesting', 'and', 'enjoyable', '.', '[SEP]']


## Handover
Record the selected model, tokenizer, number of labels, and compatibility checks for TV5.